In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0", "fqdn": "1.6.0", "h11": "0.16.0

# Независимое исследование B. Аторвастатин/HMGCR: вход, gate и валидация

Источники: [RCSB PDB 1HWK](https://www.rcsb.org/structure/1HWK), исходный кристаллический карбоксилат аторвастатина (charge −1). Координаты PDB CC0, результаты подготовки и неудача сохранены; этот карман не объявлен готовым к docking.

In [2]:
from pathlib import Path
import sys,json,hashlib,math
import numpy as np
from rdkit import Chem
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M11-L02','lab');D=DATA/'target_docking'
manifest=json.loads((D/'manifest.json').read_text(encoding='utf-8'))
for name,digest in manifest['files_sha256'].items():
    assert hashlib.sha256((D/name).read_bytes()).hexdigest()==digest,name
def read_json(name):return json.loads((D/name).read_text(encoding='utf-8'))
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight');display(fig);plt.close(fig)


## Задание 1

Из HMGCR repair audit извлеките максимальное смещение исходных pocket-heavy atoms и ближайший новый heavy atom. Сравните со строгим заранее заданным gate 0.05 Å для смещения и 6 Å для новых атомов, требующих review. Сформируйте машиночитаемый вывод: подготовка BLOCKED, docking NOT_RUN. Не создавайте фиктивные poses/scores.

In [3]:
task_1=None  # Ваш расчёт, решение и независимая проверка.


## Задание 2

На примере A2A разделите sampling и ranking, top-1 и best-by-native-RMSD, алгоритмический и научный статус. Объясните, почему выбор одного удачного seed или постфактум правильной позы создаёт смещение. Для B обоснуйте, почему этот A2A результат не заменяет target-specific redocking HMGCR.

In [4]:
task_2=None  # Ваш расчёт, решение и независимая проверка.


## Задание 3

Спроектируйте новый HMGCR redocking после научно обоснованной подготовки: независимые ligand starts, заранее заданные search seeds/критерии, symmetry-aware receptor-frame RMSD, negative translation control, сравнение cofactors/protonation и внешний комплекс. Укажите какие наблюдения разрешат запуск и какие потребуют отказа. До исправления входа score и affinity остаются NOT_RUN.

In [5]:
task_3=None  # Ваш расчёт, решение и независимая проверка.


Расчётный контракт ограничен полями: `max_original_pocket_displacement_A`, `nearest_new_heavy_atom_A`, `preparation_accepted`. Качество структурного решения оценивает преподаватель. Сохраните анализ, координатные соответствия и план в новой версии досье.

In [6]:
from research_checks import check
answer=None
print(check('M11-L02',answer))


{'status': 'NOT_RUN', 'reasoning': 'NOT_REVIEWED'}


# M11. Разбор научной статьи: от утверждения к проверке

Основная работа: **Abo Horan et al. (2025). The Chemical Stability Characterization and Kinetics of Statins in Aqueous Cyclodextrin Ocular Preparations: A Formulation Perspective**. [Полный локальный текст](../../../../../shared/data/source-responses/statin_stability_verified.xml), [издательская версия DOI](https://doi.org/10.3390/pharmaceutics17070808). Прочитайте: **Introduction / Discussion: границы биологических выводов**.

## Перенос B: самостоятельная мини-рецензия

Проследите эту ссылочную цепочку до доступного первичного свидетельства; если полного текста нет, обозначьте границу проверки. Сравните с тем, что реально проверяет redocking B.

Постройте цепь «утверждение → источник → тип наблюдения → предел переноса» и сопоставьте её с 1HWK как структурным свидетельством.

Выпишите короткое утверждение своими словами, точный раздел/Table/Figure ID, условия и единицы, способ получения данных и допустимый вывод. Помечайте отдельно опубликованные наблюдения, своё вычисление и гипотезу. Если нужных исходных точек, повторов или параметров нет, запишите «не предоставлено» и объясните, какую проверку это блокирует.

Контрольный вопрос рецензента: **Достаточны ли docking score или кристаллическая поза для вывода о тканевой экспозиции?** Ответьте со ссылкой на конкретное свидетельство, а не общим «нужно больше исследований».

## Вклад в досье и обсуждение

Сохраните `article_reviews/M11.md` внутри личного `my_research` и численную таблицу/код рядом. Используйте [шаблон мини-рецензии](../../../../../../programm/ARTICLE_REVIEW_TEMPLATE.md). Свяжите запись с вопросом и одним уже принятым решением досье: что статья подтверждает, ограничивает или заставляет изменить? Завершите планом одного независимого опыта с ожидаемым наблюдением, альтернативным исходом и критерием пересмотра.

Оценивание — [единая рубрика](../../../../../../programm/ARTICLE_REVIEW_RUBRIC.md). Ответ обсуждается с преподавателем; запуск Notebook и заполнение шаблона не означают принятия научного аргумента.